# INF0093 - Projeto Prático com Sistemas Multiagentes - 2s 2026
## Prof. Marcelo da Silva Reis
## msreis@unicamp.br

# Aula 5 - Sistemas Multiagentes
## Estudo de caso: Assistente de Análise de Editais

v2: um agente com todas as ferramentas. v3: um supervisor, dois especialistas e um sintetizador.

Mesmo documento, mesmos casos, mesmo modelo, mesmo formato de saída.

## 1. Configuração

In [ ]:
%pip install -q -U langchain langchain-groq langgraph pydantic pandas==2.2.3

In [ ]:
import os, getpass, datetime, platform, time, json, operator, re, unicodedata

def carregar_chave_groq() -> str:
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("MINHA_CHAVE_SECRETA_COLAB")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

print("Chave carregada via:", carregar_chave_groq())

In [ ]:
from langchain_groq import ChatGroq

# MODEL_NAME = "llama-3.3-70b-versatile"
MODEL_NAME = "openai/gpt-oss-20b"
TEMPERATURE = 0

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "arquitetura": "v3-supervisor",
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

## 2. Estrutura herdada

Documento, esquema de saída, casos e verificações, sem alteração.

In [ ]:
call_document = """
CHAMADA PARA PROJETOS DE INOVAÇÃO EM SISTEMAS MULTIAGENTES (AGOSTO DE 2026)

OBJETIVO
Apoiar projetos de inovação tecnológica em sistemas multiagentes, com duração
máxima de 12 meses.

ELEGIBILIDADE
Podem submeter propostas:
- pesquisadores vinculados a universidades brasileiras;
- empresas brasileiras em parceria com uma instituição de pesquisa;
- profissionais com cursos de extensão em sistemas multiagentes.

PRAZO
As propostas devem ser submetidas até 30 de outubro de 2026.

DOCUMENTOS OBRIGATÓRIOS
1. Formulário de submissão;
2. Currículo resumido do coordenador;
3. Plano de trabalho;
4. Orçamento estimado.

RESULTADO
O resultado será divulgado até 15 de dezembro de 2026.
"""

from pydantic import BaseModel, Field
from typing import Literal, Optional

class AnalysisResult(BaseModel):
    answer: str = Field(description="Resposta direta à pergunta.")
    evidence: list[str] = Field(description="Trechos literais do documento.")
    confidence: Literal["high", "medium", "low"]

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

MARCADORES_AUSENCIA = [
    "nao esta", "nao consta", "nao foi encontrad", "nao encontrei", "nao informad",
    "nao especificad", "nao ha informacao", "nao menciona", "nao e mencionad",
    "ausente no documento", "nao aparece", "nao define", "nao indica",
]

def admite_ausencia(r: AnalysisResult) -> bool:
    return (any(m in normalizar(r.answer) for m in MARCADORES_AUSENCIA)
            and len(r.evidence) == 0 and r.confidence == "low")

def cobertura_esperada(r: AnalysisResult, esperado: list) -> float:
    texto = normalizar(r.answer)
    return sum(normalizar(k) in texto for k in esperado) / len(esperado)

def evidencia_fiel(r: AnalysisResult, documento: str) -> Optional[float]:
    if not r.evidence:
        return None
    doc = normalizar(documento)
    return sum(normalizar(e) in doc for e in r.evidence) / len(r.evidence)

def avaliar(caso: dict, r: AnalysisResult) -> dict:
    if caso["verificacao"] == "manual":
        return {"aprovado": None, "cobertura": None}
    if caso["esperado"] is None:
        return {"aprovado": admite_ausencia(r), "cobertura": None}
    c = cobertura_esperada(r, caso["esperado"])
    return {"aprovado": c >= caso.get("cobertura_minima", 1.0), "cobertura": round(c, 2)}

print("[done]")

In [ ]:
test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto",
     "pergunta": "Qual é o prazo para submissão?",
     "esperado": ["30 de outubro de 2026"], "cobertura_minima": 1.0},
    {"id": "T02", "tipo": "lista", "verificacao": "auto",
     "pergunta": "Quais documentos são obrigatórios?",
     "esperado": ["formulário", "currículo", "plano de trabalho", "orçamento"],
     "cobertura_minima": 1.0},
    {"id": "T03", "tipo": "interpretação", "verificacao": "auto",
     "pergunta": "Quem pode participar?",
     "esperado": ["universidades brasileiras", "empresas brasileiras"],
     "cobertura_minima": 0.5},
    {"id": "T04", "tipo": "informação ausente", "verificacao": "auto",
     "pergunta": "Qual é o valor máximo de financiamento?", "esperado": None},
    {"id": "T05", "tipo": "ambíguo", "verificacao": "manual",
     "pergunta": "Qual é o prazo?", "esperado": None},
    {"id": "T06", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar e qual é o prazo para submissão?",
     "esperado": ["universidades brasileiras", "30 de outubro de 2026"],
     "cobertura_minima": 1.0},

    # Novo na Aula 5: exige as duas responsabilidades e mais um item.
    #
    {"id": "T07", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar, quais documentos preciso e até quando envio?",
     "esperado": ["universidades brasileiras", "plano de trabalho", "30 de outubro de 2026"],
     "cobertura_minima": 1.0},
]

print(len(test_cases), "casos")

## 3. Ferramentas

In [ ]:
from langchain_core.tools import tool

def _secao(documento: str, titulo: str) -> str:
    capturando, coletado = False, []
    for linha in documento.strip().split("\n"):
        if linha.strip().isupper() and len(linha.strip()) > 3:
            if capturando:
                break
            capturando = normalizar(titulo) in normalizar(linha)
            continue
        if capturando and linha.strip():
            coletado.append(linha.strip())
    return "\n".join(coletado)

@tool
def consultar_elegibilidade() -> str:
    """Trecho do edital sobre quem pode submeter propostas."""
    return _secao(call_document, "ELEGIBILIDADE") or "Seção não encontrada."

@tool
def consultar_prazo() -> str:
    """Trecho do edital sobre prazos de submissão."""
    return _secao(call_document, "PRAZO") or "Seção não encontrada."

@tool
def consultar_documentos() -> str:
    """Lista de documentos obrigatórios exigidos pelo edital."""
    return _secao(call_document, "DOCUMENTOS OBRIGATÓRIOS") or "Seção não encontrada."

@tool
def consultar_resultado() -> str:
    """Trecho do edital sobre divulgação de resultados."""
    return _secao(call_document, "RESULTADO") or "Seção não encontrada."

todas_tools = [consultar_elegibilidade, consultar_prazo, consultar_documentos, consultar_resultado]
print("[done]")

## 4. v2 de referência

Um agente, todas as ferramentas. Reexecutado hoje, com o mesmo modelo da v3.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class EstadoAgente(TypedDict):
    messages: Annotated[list, add_messages]

def montar_agente(instrucao: str, tools: list):
    """Monta o laço agente-ferramentas da Aula 3. Um especialista é isso com escopo menor."""
    ligado = llm.bind_tools(tools)

    def no_agente(state: EstadoAgente):
        return {"messages": [ligado.invoke([SystemMessage(content=instrucao)] + state["messages"])]}

    b = StateGraph(EstadoAgente)
    b.add_node("agente", no_agente)
    b.add_node("tools", ToolNode(tools))
    b.add_edge(START, "agente")
    b.add_conditional_edges("agente", tools_condition)
    b.add_edge("tools", "agente")
    return b.compile()

INSTRUCAO_V2 = """
Você é um assistente de análise de editais.
Use as ferramentas para obter os trechos necessários.
Para perguntas compostas, consulte TODAS as informações necessárias antes de responder.
Não invente informações.
"""

agente_v2 = montar_agente(INSTRUCAO_V2, todas_tools)
print("[v2 pronta]")

In [ ]:
SYSTEM_FORMAT = """
Converta a análise abaixo no formato estruturado.
'evidence' deve conter apenas trechos copiados literalmente do documento.
Se a informação não constar, diga isso na 'answer', deixe 'evidence' vazia e use 'confidence' baixa.
"""

estruturado = llm.with_structured_output(AnalysisResult, include_raw=True)

def sintetizar(pergunta: str, material: str):
    saida = estruturado.invoke(
        SYSTEM_FORMAT + "\n\nDOCUMENTO:\n" + call_document
        + "\n\nPERGUNTA:\n" + pergunta + "\n\nANÁLISE:\n" + material
    )
    return saida["parsed"]

def contar(mensagens: list) -> dict:
    return {
        "chamadas_llm": sum(1 for m in mensagens if isinstance(m, AIMessage)),
        "chamadas_tool": sum(len(getattr(m, "tool_calls", None) or []) for m in mensagens),
        "erros_tool": sum(1 for m in mensagens
                          if isinstance(m, ToolMessage) and getattr(m, "status", None) == "error"),
    }

def responder_v2(pergunta: str, limite: int = 12):
    inicio = time.perf_counter()
    estado = agente_v2.invoke({"messages": [HumanMessage(content=pergunta)]},
                              config={"recursion_limit": limite})
    metricas = contar(estado["messages"])
    material = "\n".join(str(getattr(m, "content", "")) for m in estado["messages"])
    resultado = sintetizar(pergunta, material)
    metricas["chamadas_llm"] += 1
    metricas["latencia_s"] = round(time.perf_counter() - inicio, 2)
    return resultado, metricas

print("[done]")

## 5. v3: supervisor e especialistas

Dois especialistas, cada um com instrução curta e apenas as ferramentas do seu domínio.

In [ ]:
ESPECIALISTAS = {
    "elegibilidade": {
        "instrucao": ("Você responde apenas sobre quem pode submeter propostas e sob que condições. "
                      "Use a ferramenta disponível e cite trechos literais. "
                      "Se a pergunta não for do seu escopo, diga que está fora do escopo."),
        "tools": [consultar_elegibilidade],
    },
    "prazos_documentos": {
        "instrucao": ("Você responde apenas sobre datas, prazos, documentos obrigatórios e "
                      "divulgação de resultados. Use as ferramentas disponíveis e cite trechos "
                      "literais. Se a pergunta não for do seu escopo, diga que está fora do escopo."),
        "tools": [consultar_prazo, consultar_documentos, consultar_resultado],
    },
}

agentes = {nome: montar_agente(cfg["instrucao"], cfg["tools"])
           for nome, cfg in ESPECIALISTAS.items()}

print("Especialistas:", list(agentes))

O supervisor decide o próximo passo com saída estruturada, escolhendo entre os especialistas ainda
não acionados ou o encerramento. Restringir as opções em tempo de execução evita que ele repita um
especialista indefinidamente.

In [ ]:
class Decisao(BaseModel):
    proximo: str = Field(description="Nome do especialista a acionar, ou 'sintetizar'.")
    motivo: str = Field(description="Uma frase justificando a escolha.")

# Instanciando o supervisor.
#
decisor = llm.with_structured_output(Decisao)

class EstadoMAS(TypedDict):
    pergunta: str
    achados: Annotated[list, operator.add]
    trace: Annotated[list, operator.add]
    visitados: Annotated[list, operator.add]
    proximo: str
    resultado: Optional[AnalysisResult]


# Nó de execução do supervisor.
#
def no_supervisor(state: EstadoMAS):

    disponiveis = [n for n in agentes if n not in state["visitados"]]
    if not disponiveis:
        return {"proximo": "sintetizar",
                "trace": [{"agente": "supervisor", "decisao": "sintetizar",
                           "motivo": "todos os especialistas já foram acionados"}]}

    inicio = time.perf_counter()

    catalogo = "\n".join(f"- {n}: {ESPECIALISTAS[n]['instrucao'][:90]}" for n in disponiveis)

    ja = "\n".join(f"- {a['agente']}: {a['conteudo'][:200]}" for a in state["achados"]) or "(nada ainda)"

    d = decisor.invoke(
        "Você coordena especialistas. Escolha o próximo a acionar, ou 'sintetizar' se já há "
        "informação suficiente para responder por completo.\n\n"
        f"PERGUNTA: {state['pergunta']}\n\nDISPONÍVEIS:\n{catalogo}\n\nJÁ COLETADO:\n{ja}"
    )

    escolha = d.proximo if d.proximo in disponiveis else "sintetizar"

    return {"proximo": escolha,
            "trace": [{"agente": "supervisor", "decisao": escolha, "motivo": d.motivo,
                       "latencia_s": round(time.perf_counter() - inicio, 2),
                       "chamadas_llm": 1, "chamadas_tool": 0}]}


# Nó de execução de especialista.
#
def fazer_no_especialista(nome: str):
    def no(state: EstadoMAS):

        inicio = time.perf_counter()

        estado = agentes[nome].invoke(
            {"messages": [HumanMessage(content=state["pergunta"])]},
            config={"recursion_limit": 8},
        )
        m = contar(estado["messages"])
        conteudo = estado["messages"][-1].content
        return {
            "achados": [{"agente": nome, "conteudo": conteudo}],
            "visitados": [nome],
            "trace": [{"agente": nome, "latencia_s": round(time.perf_counter() - inicio, 2), **m}],
        }
    return no


# Nó de execução do sintetizador.
#
def no_sintetizador(state: EstadoMAS):

    inicio = time.perf_counter()

    material = "\n\n".join(f"[{a['agente']}]\n{a['conteudo']}" for a in state["achados"])
    resultado = sintetizar(state["pergunta"], material or "(nenhum achado)")
    return {"resultado": resultado,
            "trace": [{"agente": "sintetizador",
                       "latencia_s": round(time.perf_counter() - inicio, 2),
                       "chamadas_llm": 1, "chamadas_tool": 0}]}

print("[done]")

In [ ]:
b = StateGraph(EstadoMAS)

b.add_node("supervisor", no_supervisor)

for nome in agentes:
    b.add_node(nome, fazer_no_especialista(nome))

b.add_node("sintetizar", no_sintetizador)

b.add_edge(START, "supervisor")
b.add_conditional_edges("supervisor", lambda s: s["proximo"],
                        {**{n: n for n in agentes}, "sintetizar": "sintetizar"})
for nome in agentes:
    b.add_edge(nome, "supervisor")

b.add_edge("sintetizar", END)

app_v3 = b.compile()

def responder_v3(pergunta: str, limite: int = 20):
    inicio = time.perf_counter()
    estado = app_v3.invoke(
        {"pergunta": pergunta, "achados": [], "trace": [], "visitados": [],
         "proximo": "", "resultado": None},
        config={"recursion_limit": limite},
    )
    trace = estado["trace"]
    metricas = {
        "chamadas_llm": sum(t.get("chamadas_llm", 0) for t in trace),
        "chamadas_tool": sum(t.get("chamadas_tool", 0) for t in trace),
        "erros_tool": sum(t.get("erros_tool", 0) for t in trace),
        "agentes": [t["agente"] for t in trace if t["agente"] in agentes],
        "latencia_s": round(time.perf_counter() - inicio, 2),
    }
    return estado["resultado"], metricas, trace

print("[v3 pronta]")

In [ ]:
# Visualização do grafo do v2 (opcional, requer IPython).
#
try:
    from IPython.display import Image, display
    print("Nosso v2:")
    display(Image(agente_v2.get_graph().draw_mermaid_png()))
except Exception as erro:
    print("Sem renderização de imagem:", type(erro).__name__)
    print(agente_v2.get_graph().draw_ascii())

print()
print()

# Visualização do grafo do v3.
#
try:
    from IPython.display import Image, display
    print("Nosso v3:")
    display(Image(app_v3.get_graph().draw_mermaid_png()))
except Exception as erro:
    print("Sem renderização de imagem:", type(erro).__name__)
    print(app_v3.get_graph().draw_ascii())


## 6. Uma pergunta composta

In [ ]:
resultado, metricas, trace = responder_v3(
    "Quem pode participar, quais documentos preciso e até quando envio?")

print("RESPOSTA :", resultado.answer)
print("MÉTRICAS :", metricas)
print()
for t in trace:
    print(" ", t["agente"], "|", {k: v for k, v in t.items() if k != "agente"})

A rota é o resultado mais informativo aqui. Ela mostra se o supervisor acionou os dois especialistas
ou tentou responder com um só.

## 7. Comparação v2 e v3

In [ ]:
registros = []

for caso in test_cases:
    r2, m2 = responder_v2(caso["pergunta"])
    a2 = avaliar(caso, r2)
    r3, m3, t3 = responder_v3(caso["pergunta"])
    a3 = avaliar(caso, r3)

    registros.append({
        "id": caso["id"], "tipo": caso["tipo"],
        "v2_aprovado": a2["aprovado"], "v3_aprovado": a3["aprovado"],
        "v2_evid": evidencia_fiel(r2, call_document),
        "v3_evid": evidencia_fiel(r3, call_document),
        "v2_latencia": m2["latencia_s"], "v3_latencia": m3["latencia_s"],
        "v2_llm": m2["chamadas_llm"], "v3_llm": m3["chamadas_llm"],
        "v2_tools": m2["chamadas_tool"], "v3_tools": m3["chamadas_tool"],
        "rota": " > ".join(m3["agentes"]) or "(nenhum)",
        "v2_resposta": r2.answer, "v3_resposta": r3.answer,
    })
    print(f'[{caso["id"]}] v2={a2["aprovado"]} v3={a3["aprovado"]} | rota: {registros[-1]["rota"]}')

In [ ]:
import pandas as pd

df = pd.DataFrame(registros)
df[["id", "tipo", "v2_aprovado", "v3_aprovado", "rota",
    "v2_latencia", "v3_latencia", "v2_llm", "v3_llm"]]

In [ ]:
autos = df[df["v2_aprovado"].notna()]

COMPARACAO = {
    "casos_automaticos": int(len(autos)),
    "v2_taxa": round(float(autos["v2_aprovado"].astype(bool).mean()), 2),
    "v3_taxa": round(float(autos["v3_aprovado"].astype(bool).mean()), 2),
    "v2_latencia_mediana_s": round(float(df["v2_latencia"].median()), 2),
    "v3_latencia_mediana_s": round(float(df["v3_latencia"].median()), 2),
    "v2_chamadas_llm": int(df["v2_llm"].sum()),
    "v3_chamadas_llm": int(df["v3_llm"].sum()),
    "v2_chamadas_tool": int(df["v2_tools"].sum()),
    "v3_chamadas_tool": int(df["v3_tools"].sum()),
}
COMPARACAO

In [ ]:
referencia = {"run": RUN_INFO, "comparacao": COMPARACAO, "registros": registros}
with open("v3_vs_v2_resultados.json", "w", encoding="utf-8") as f:
    json.dump(referencia, f, ensure_ascii=False, indent=2, default=str)
print("Salvo em v3_vs_v2_resultados.json")

## 8. Discussão

1. A divisão mudou a taxa de acerto, ou apenas o custo?
2. Em que casos o supervisor acionou um especialista desnecessário?
3. Algum achado correto se perdeu na síntese?
4. O contrato entre especialista e supervisor está claro, ou o sintetizador precisa reler o documento?
5. Que terceiro especialista o problema pediria, e com que evidência?


## 9. Exercício

1. Identifique no seu sistema duas responsabilidades que hoje competem no mesmo prompt.
2. Separe-as, mantendo ferramentas e instruções por escopo.
3. Escolha um padrão de organização e justifique.
4. Registre a rota de cada caso.
5. Reexecute v2 e o v3 (parcial) sobre o conjunto congelado, no mesmo modelo.
6. Responda: a divisão se pagou?